# الفصل 6: التقطيع إلى وحدات: Tokenization

> قبل أن يرى أي نموذج لغوي كلمة واحدة، يجب تحويل النص إلى تسلسل من الأرقام الصحيحة (Token IDs).

- **الجزء:** 2 — التمثيل اللغوي
- **الفصل:** 6
- **المستوى:** متوسط
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 5: قياس الأداء: مقاييس التصنيف والحيرة
- **الوسوم:** BPE، WordPiece، SentencePiece، Tokenizer

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** sentencepiece
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("sentencepiece", "sentencepiece")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch06-tokenization.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch06-tokenization.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل أن يرى أي نموذج لغوي كلمة واحدة، يجب تحويل النص إلى تسلسل من الأرقام الصحيحة (Token IDs).

> **بيانات الفصل**
>
> - **الجزء:** 2 — التمثيل اللغوي
> - **المستوى:** متوسط
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 5: قياس الأداء: مقاييس التصنيف والحيرة


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم إعدادات SentencePiece الفعلية للعربية
- فهم BPE
- فهم WordPiece


## 1. لماذا يهمّك هذا الموضوع؟

قبل أن يرى أي نموذج لغوي كلمة واحدة، يجب تحويل النص إلى تسلسل من الأرقام الصحيحة (Token IDs). القرار الذي يبدو بسيطًا — "كيف نُقسِّم النص؟" — يُحدِّد فعليًا: حجم المفردات، كفاءة النموذج في التعامل مع الكلمات النادرة والمُركَّبة، تكلفة الحوسبة (عدد الرموز = عدد خطوات المعالجة)، وحتى **عدالة النموذج بين اللغات المختلفة**. هذا مهم بشكل خاص للعربية، التي تتميز بثراء صرفي هائل (كلمة واحدة قد تحمل جذرًا ووزنًا وضمائر متصلة ولواحق).


## 2. الخريطة الذهنية


```text
مستويات التقطيع الممكنة:

Character-level      Word-level         Subword-level (الحل الوسط الفائز)
   "كتاب" → [ك,ت,ا,ب]   "كتاب" → [كتاب]    "كتاب" → [كتاب] أو [كت, اب] حسب التكرار
   مفردات صغيرة جدًا     مفردات ضخمة جدًا   مفردات معتدلة + يتعامل مع الكلمات النادرة
   تسلسلات طويلة جدًا    مشكلة OOV حادة     التوازن الأمثل المُستخدَم في كل LLM حديث
                                              │
                          ┌───────────────────┼───────────────────┐
                          ▼                   ▼                   ▼
                        BPE              WordPiece          SentencePiece/Unigram
                   (GPT, Llama)             (BERT)            (T5, Llama, multilingual)
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل أنك تريد بناء "قاموس" محدود الحجم (مثلًا 50,000 رمز) يُغطّي أي نص ممكن في أي لغة. إذا استخدمت الكلمات الكاملة فقط، ستحتاج ملايين المداخل (ولن تُغطّي كلمات جديدة لم ترَها أثناء البناء — مشكلة Out-Of-Vocabulary). إذا استخدمت الأحرف فقط، ستُغطّي كل شيء لكن كل جملة ستصبح تسلسلًا طويلًا جدًا من الرموز (معالجة أبطأ، وصعوبة أكبر في التقاط المعنى). **Subword Tokenization** هو الحل الوسط الذكي: الكلمات الشائعة جدًا تبقى كاملة (رمز واحد)، بينما الكلمات النادرة أو المُركَّبة تُقسَّم إلى أجزاء أصغر متكررة الاستخدام.

### 🟡 طبقة الآلية — خوارزمية BPE (Byte-Pair Encoding)

الفكرة: نبدأ بتقسيم النص إلى أحرف منفردة، ثم **نُدمِج تكراريًا** أكثر زوج متجاور تكرارًا في كل خطوة، حتى نصل للحجم المطلوب للمفردات.

### 🔴 طبقة التعمّق — BPE من الصفر بالكامل


In [ ]:
import re
from collections import defaultdict, Counter

class BPETokenizerFromScratch:
    """
    تطبيق كامل لخوارزمية Byte-Pair Encoding (Sennrich et al., 2016)
    من الصفر، بدون أي مكتبة جاهزة. هذا هو جوهر Tokenizer في
    GPT-2/GPT-3/GPT-4 وLlama (مع بعض التحسينات الهندسية الإضافية).
    """
    def __init__(self, vocab_size: int = 1000):
        self.vocab_size = vocab_size
        self.merges = {}       # قاموس الدمجات المُتعلَّمة: (رمز1, رمز2) -> رمز مُدمَج
        self.vocab = {}        # قاموس المفردات النهائي: رمز -> معرّف رقمي

    def _get_word_frequencies(self, corpus: list) -> dict:
        """
        حساب تكرار كل كلمة في المجموعة النصية، مع تمثيل كل كلمة
        كتسلسل أحرف مُنفصلة + رمز نهاية الكلمة </w> (ضروري لتمييز
        "نهاية الكلمة" عن استمرارها، فـ "كتاب" و"كتابة" تشتركان
        في البادئة لكن يجب تمييز حدود الكلمة الفعلية).
        """
        word_freqs = defaultdict(int)
        for text in corpus:
            words = text.strip().split()
            for word in words:
                # تمثيل الكلمة كتسلسل أحرف منفصلة بمسافات + رمز نهاية
                char_sequence = " ".join(list(word)) + " </w>"
                word_freqs[char_sequence] += 1
        return word_freqs

    def _get_pair_frequencies(self, word_freqs: dict) -> dict:
        """حساب تكرار كل زوج أحرف/رموز متجاورة عبر كل الكلمات."""
        pair_freqs = defaultdict(int)
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                pair_freqs[pair] += freq
        return pair_freqs

    def _merge_pair(self, pair: tuple, word_freqs: dict) -> dict:
        """
        دمج زوج رموز مُحدَّد في كل الكلمات التي يظهر فيها.
        مثال: دمج ('ك','ت') يُحوِّل "ك ت ا ب" إلى "كت ا ب"
        """
        new_word_freqs = {}
        bigram = re.escape(" ".join(pair))
        pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")

        for word, freq in word_freqs.items():
            new_word = pattern.sub("".join(pair), word)
            new_word_freqs[new_word] = freq
        return new_word_freqs

    def train(self, corpus: list, verbose: bool = True):
        """
        حلقة التدريب الرئيسية: نُكرِّر "إيجاد أكثر زوج تكرارًا ودمجه"
        حتى نصل لحجم المفردات المطلوب أو لا تبقى أزواج قابلة للدمج.
        """
        word_freqs = self._get_word_frequencies(corpus)

        # المفردات الابتدائية: كل حرف فريد + رمز نهاية الكلمة
        initial_chars = set()
        for word in word_freqs:
            initial_chars.update(word.split())
        self.vocab = {char: idx for idx, char in enumerate(sorted(initial_chars))}

        num_merges = self.vocab_size - len(self.vocab)
        for merge_step in range(num_merges):
            pair_freqs = self._get_pair_frequencies(word_freqs)
            if not pair_freqs:
                break  # لا توجد أزواج أخرى للدمج

            best_pair = max(pair_freqs, key=pair_freqs.get)
            word_freqs = self._merge_pair(best_pair, word_freqs)

            merged_token = "".join(best_pair)
            self.merges[best_pair] = merged_token
            self.vocab[merged_token] = len(self.vocab)

            if verbose and merge_step % 50 == 0:
                print(f"الدمج {merge_step}: {best_pair} → '{merged_token}' "
                      f"(تكرار={pair_freqs[best_pair]})")

        if verbose:
            print(f"\n✅ اكتمل التدريب: {len(self.vocab)} رمز في المفردات")

    def tokenize(self, text: str) -> list:
        """تقطيع نص جديد باستخدام الدمجات المُتعلَّمة، بنفس الترتيب الذي تعلّمناها به."""
        words = text.strip().split()
        tokens = []

        for word in words:
            symbols = list(word) + ["</w>"]
            # تطبيق الدمجات المُتعلَّمة تكراريًا بنفس ترتيب الأولوية
            while len(symbols) > 1:
                pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
                # إيجاد الدمج ذي الأولوية الأعلى (الذي تعلّمناه أولًا)
                mergeable = [p for p in pairs if p in self.merges]
                if not mergeable:
                    break
                # نختار أول دمج تعلّمناه (الأقدم زمنيًا = الأكثر عمومية/تكرارًا)
                pair_to_merge = min(mergeable, key=lambda p: list(self.merges.keys()).index(p))
                idx = pairs.index(pair_to_merge)
                symbols = (symbols[:idx] + [self.merges[pair_to_merge]] +
                           symbols[idx + 2:])
            tokens.extend(symbols)

        return tokens

    def encode(self, text: str) -> list:
        """تحويل النص إلى تسلسل معرّفات رقمية (Token IDs)."""
        tokens = self.tokenize(text)
        return [self.vocab.get(tok, self.vocab.get("<unk>", -1)) for tok in tokens]

# === تجربة عملية على نصوص عربية ===
arabic_corpus = [
    "الكتاب على الطاولة",
    "الكتابة مهارة مهمة",
    "كتب الطالب الدرس",
    "يكتب المعلم على السبورة",
    "الكاتب نشر كتابا جديدا",
] * 20  # تكرار لمحاكاة بيانات أكبر

tokenizer = BPETokenizerFromScratch(vocab_size=120)
tokenizer.train(arabic_corpus, verbose=True)

test_text = "الكاتب يكتب كتابا"
tokens = tokenizer.tokenize(test_text)
print(f"\nالنص: {test_text}")
print(f"الرموز الناتجة: {tokens}")
# الرموز الناتجة: ['الكاتب</w>', 'يكتب</w>', 'كتاب', 'ا</w>']
# لاحظ كيف أن الجذر المشترك "كتب/كتاب" يظهر كوحدة متكررة عبر عدة كلمات


### مقارنة WordPiece (BERT) وSentencePiece/Unigram (T5, Llama)


In [ ]:
def explain_wordpiece_difference():
    """
    WordPiece (مُستخدَمة في BERT): تشبه BPE لكن معيار اختيار الزوج
    للدمج مختلف جوهريًا. بدلًا من "أكثر زوج تكرارًا" كما في BPE،
    تختار WordPiece الزوج الذي يُعظِّم احتمالية المجموعة التدريبية
    عند دمجه (Likelihood-based)، أي:

        score(pair) = freq(pair) / (freq(first) × freq(second))

    هذا يُفضِّل دمج الأزواج النادرة التي تظهر معًا بشكل "غير متوقع"
    (أعلى من الصدفة الإحصائية)، وليس فقط الأكثر تكرارًا بشكل مطلق.
    """
    # محاكاة الفرق: زوج ('ال', 'كتاب') قد يكون أقل تكرارًا من ('ت','ا')
    # لكن احتمالية ظهورهما معًا أعلى بكثير من المتوقع عشوائيًا
    pair_freq = 50
    first_freq, second_freq = 200, 180

    bpe_score = pair_freq  # BPE يهتم فقط بالتكرار المطلق
    wordpiece_score = pair_freq / (first_freq * second_freq)

    print(f"BPE score (تكرار مطلق): {bpe_score}")
    print(f"WordPiece score (احتمالية مُطبَّعة): {wordpiece_score:.6f}")

def explain_unigram_sentencepiece():
    """
    SentencePiece/Unigram (T5, Llama, XLNet): نهج معاكس تمامًا!
    بدلًا من البدء بأحرف والدمج تصاعديًا (Bottom-up كما في BPE/WordPiece)،
    نبدأ بمفردات كبيرة جدًا (كل الكلمات الفرعية المحتملة تقريبًا)
    ثم نُقلِّص تدريجيًا (Top-down) عبر إزالة الرموز التي تُقلِّل
    إزالتها أقل احتمالية ممكنة من إجمالي احتمالية المجموعة التدريبية
    (نموذج احتمالي Unigram Language Model كامل).

    الميزة الحاسمة: SentencePiece يُعامل النص كـ"تيار بايتات خام"
    (raw byte stream) بما في ذلك المسافات (تُمثَّل برمز خاص ▁)،
    مما يجعله مستقلًا تمامًا عن قواعد التقطيع بلغة معينة —
    ميزة بالغة الأهمية للنماذج متعددة اللغات (بما فيها العربية
    التي لا تحتاج فصل كلمات بالطريقة اللاتينية نفسها دائمًا).
    """
    example = "الكتاب جميل"
    sentencepiece_style = "▁الكتاب▁جميل"  # ▁ يُمثِّل المسافة كرمز صريح
    print(f"تمثيل SentencePiece: {sentencepiece_style}")
    print("هذا يسمح بإعادة بناء النص الأصلي بدقة تامة دون أي قواعد")
    print("خاصة باللغة لتحديد أين تقع المسافات (reversible tokenization)")


### إعدادات SentencePiece الفعلية للعربية


In [ ]:
import sentencepiece as spm

def train_sentencepiece_for_arabic(corpus_file: str, model_prefix: str = "arabic_tokenizer"):
    """
    التدريب الفعلي لـ SentencePiece على بيانات عربية، باستخدام
    إعدادات مُحسَّنة خصيصًا للتعامل مع خصائص العربية (التشكيل،
    الحروف المُركَّبة، النطاق الواسع من الرموز اليونيكود).
    """
    spm.SentencePieceTrainer.train(
        input=corpus_file,
        model_prefix=model_prefix,
        vocab_size=32000,
        model_type="unigram",        # أو "bpe" حسب التفضيل

        # إعداد حرج جدًا للعربية: character_coverage يُحدِّد نسبة
        # الأحرف التي يجب أن يُغطّيها النموذج. للعربية (ولغات أخرى
        # غنية بالرموز مثل التشكيل والحروف الخاصة)، القيمة الافتراضية
        # 0.9995 (بدلًا من 1.0) تُهمل الأحرف النادرة جدًا (أخطاء إملائية،
        # رموز شاذة) لتجنّب تضخيم المفردات بأحرف غير مفيدة عمليًا.
        character_coverage=0.9995,

        normalization_rule_name="nfkc",  # تطبيع يونيكود موحّد (مهم جدًا للعربية
                                           # بسبب تعدد أشكال نفس الحرف Unicode)
        byte_fallback=True,   # أي رمز غير مُغطّى يُرمَّز كبايتات خام (لا OOV إطلاقًا!)

        # رموز خاصة ضرورية لأي LLM
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols=["<|system|>", "<|user|>", "<|assistant|>"],
    )
    print(f"✅ تم تدريب SentencePiece Tokenizer وحفظه بـ {model_prefix}.model")


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: لديك مجموعة نصية صغيرة جدًا: `["low", "lower", "lowest", "newest", "widest"]` (مثال كلاسيكي من ورقة BPE الأصلية). طبّق أول 3 خطوات دمج يدويًا.


In [ ]:
def manual_bpe_walkthrough():
    corpus = {
        "l o w </w>": 5,
        "l o w e r </w>": 2,
        "l o w e s t </w>": 6,
        "n e w e s t </w>": 3,
        "w i d e s t </w>": 2,
    }

    for step in range(3):
        pair_freqs = defaultdict(int)
        for word, freq in corpus.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pair_freqs[(symbols[i], symbols[i+1])] += freq

        best_pair = max(pair_freqs, key=pair_freqs.get)
        print(f"الخطوة {step+1}: أكثر زوج تكرارًا = {best_pair} "
              f"(تكرار={pair_freqs[best_pair]})")

        new_corpus = {}
        merged = "".join(best_pair)
        for word, freq in corpus.items():
            new_word = word.replace(" ".join(best_pair), merged)
            new_corpus[new_word] = freq
        corpus = new_corpus
        print(f"   الكلمات بعد الدمج: {list(corpus.keys())}\n")

manual_bpe_walkthrough()
# الخطوة 1: أكثر زوج تكرارًا = ('e', 's') (تكرار=9)
#    الكلمات بعد الدمج: ['l o w </w>', 'l o w e r </w>', 'l o w es t </w>', ...]
# الخطوة 2: أكثر زوج تكرارًا = ('es', 't') (تكرار=9)
#    الكلمات بعد الدمج: [..., 'l o w est </w>', 'n e w est </w>', ...]
# الخطوة 3: أكثر زوج تكرارًا = ('l', 'o') (تكرار=13)
#    الكلمات بعد الدمج: ['lo w </w>', 'lo w e r </w>', 'lo w est </w>', ...]


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم `character_coverage≈0.9995` للعربية | يوازن بين تغطية شاملة وتجنّب تضخّم المفردات بأحرف نادرة جدًا |
| طبّق تطبيع NFC/NFKC دائمًا للعربية | يوحّد الأشكال المختلفة لنفس الحرف (مثل الألف بصورها المختلفة) قبل التدريب |
| فعّل `byte_fallback=True` | يضمن عدم وجود أي رمز "غير معروف" (OOV) إطلاقًا — أي نص قابل للترميز دائمًا |
| قِس "Fertility" للغتك | متوسط عدد الرموز لكل كلمة؛ قيمة عالية تعني Tokenizer غير مُحسَّن للغتك (الفصل 19 يتعمّق أكثر) |


## 6. أخطاء شائعة ومصائد

- **تجاهل تطبيع Unicode قبل التدريب**: نفس الكلمة العربية قد تُمثَّل بعدة تسلسلات بايت مختلفة (خصوصًا مع الهمزات والتشكيل)، مما يُضاعف حجم المفردات دون داعٍ ويُشتِّت الإحصائيات.
- **استخدام Tokenizer مُدرَّب على الإنجليزية فقط لنص عربي**: يؤدي لتقطيع العربية إلى بايتات فردية شبه عشوائية، مما يُضاعف طول التسلسل ويُضعف الأداء بشكل كبير (مشكلة شائعة جدًا في النماذج الأولى غير متعددة اللغات).
- **الخلط بين حجم المفردات الكبير جدًا والأداء الأفضل**: مفردات أكبر تعني embedding matrix أكبر (تكلفة ذاكرة)، وقد لا تتحسن الكفاءة اللغوية الفعلية بعد نقطة معينة.


## 7. جدول مقارنة

| الخوارزمية | معيار الدمج/الاختيار | الاتجاه | أمثلة نماذج |
|---|---|---|---|
| BPE | أكثر زوج تكرارًا (تكرار مطلق) | تصاعدي (Bottom-up) | GPT-2/3/4, Llama (أصليًا) |
| WordPiece | أعلى احتمالية مُطبَّعة | تصاعدي | BERT, DistilBERT |
| Unigram (SentencePiece) | أقل خسارة احتمالية عند الإزالة | تنازلي (Top-down) | T5, XLNet, Llama (حديثًا), Gemini |
| Character-level | لا يوجد (كل حرف رمز) | — | نماذج بحثية متخصصة، BLT (الفصل 62) |


## 8. ملخص أفضل الممارسات

- استخدم SentencePiece مع `unigram` كخيار افتراضي قوي للنماذج متعددة اللغات بما فيها العربية.
- طبّق تطبيع Unicode (NFC/NFKC) دائمًا قبل التدريب.
- فعّل `byte_fallback` لضمان عدم وجود مشكلة OOV إطلاقًا.
- قِس Fertility على عيّنة من لغتك المستهدفة قبل اعتماد أي Tokenizer جاهز.


## 9. أسئلة مقابلات

**س: لماذا تُفضَّل Subword Tokenization (مثل BPE) على التقطيع على مستوى الكلمة الكاملة في النماذج الحديثة؟**

**ج**: ثلاثة أسباب رئيسية: (1) **مشكلة OOV**: التقطيع على مستوى الكلمة يحتاج مفردات ضخمة جدًا لتغطية كل الكلمات الممكنة، ويفشل تمامًا مع أي كلمة جديدة لم تظهر في بيانات التدريب (أسماء أعلام، كلمات تقنية جديدة، أخطاء إملائية). (2) **كفاءة التمثيل**: Subword Tokenization تسمح بتمثيل الكلمات المُركَّبة (شائعة جدًا في العربية والألمانية) عبر إعادة استخدام أجزاء شائعة، فتُقلِّل حجم المفردات المطلوب مع الحفاظ على تغطية شاملة. (3) **التعميم المورفولوجي**: كلمات مُشتقّة من نفس الجذر (مثل "كتب"، "كاتب"، "مكتوب"، "كتابة") تُقسَّم غالبًا بشكل يُحافظ على الجذر المشترك كوحدة واحدة، مما يساعد النموذج على تعلّم العلاقات الصرفية بكفاءة أعلى من التعامل مع كل كلمة كوحدة منفصلة تمامًا.

**س: ما الفرق الجوهري بين نهج BPE/WordPiece (Bottom-up) ونهج Unigram (Top-down)؟**

**ج**: BPE وWordPiece يبدآن من أصغر وحدة (الأحرف المنفردة) ويدمجان تصاعديًا بناءً على معيار محلي (تكرار الزوج في BPE، أو احتمالية مُطبَّعة في WordPiece) في كل خطوة، وهذا قرار **جشع (greedy)** لا يراجع نفسه لاحقًا. أما Unigram فيبدأ من مفردات كبيرة جدًا (كل الكلمات الفرعية المحتملة تقريبًا) ثم يُقلِّصها تدريجيًا عبر حذف الرموز التي يُسبِّب حذفها أقل انخفاض في احتمالية (likelihood) المجموعة التدريبية الكاملة بموجب نموذج احتمالي Unigram صريح. هذا النهج "الشمولي" (holistic) يسمح لـ Unigram بإعادة تقييم جودة كل رمز في سياق المفردات الكاملة، وغالبًا ما يُنتج تقطيعًا أكثر مرونة (يمكنه توليد عدة تقطيعات محتملة لنفس الكلمة مع احتمالاتها، مفيد لتقنيات مثل Subword Regularization في التدريب).


## 10. تمرين عملي

درّب `BPETokenizerFromScratch` بحجم مفردات 300 على نص عربي حقيقي (فقرة من ويكيبيديا العربية، 500+ كلمة)، ثم قارن متوسط عدد الرموز لكل كلمة (Fertility) مع تطبيق نفس الفقرة عبر `tiktoken` (Tokenizer الخاص بـ GPT-4) وسجّل الفرق — سيُظهر لك هذا عمليًا كيف أن Tokenizers غير المُحسَّنة للعربية تُنتج تسلسلات أطول بكثير.


## 11. مراجع للتعمّق

- Sennrich, Haddow & Birch (2016) — *Neural Machine Translation of Rare Words with Subword Units* (BPE)
- Schuster & Nakajima (2012) — *Japanese and Korean Voice Search* (WordPiece)
- Kudo (2018) — *Subword Regularization* + Kudo & Richardson (2018) — *SentencePiece*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التمثيل اللغوي / الفصل 6: التقطيع إلى وحدات: Tokenization.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
